# TSP — 32 Capitales de México: Colonias de Hormigas y Algoritmo Genético

Este notebook resuelve el **Problema del Agente Viajero (TSP)** para las 32 capitales
estatales de México usando dos metaheurísticas de naturaleza bio-inspirada.
El objetivo es minimizar el **costo total de viaje** (combustible + peajes + tiempo
del vendedor) visitando exactamente una vez cada capital y regresando al punto de partida.

| # | Sección | Contenido |
|---|---------|----------|
| 1 | Datos y modelo de costo | 32 capitales, distancia haversine, función objetivo |
| 2 | Colonias de Hormigas (ACO) | Teoría, implementación, animación de convergencia |
| 3 | Algoritmo Genético (GA) | Teoría, OX crossover, animación de convergencia |
| 4 | Experimentos: 30 corridas | Comparativa estadística ACO vs GA |
| 5 | Mejor ruta | Visualización sobre mapa de México |
| 6 | Conclusiones | Hallazgos y recomendaciones |

In [ ]:
# !pip install numpy matplotlib pillow deap

In [ ]:
# ── Imports y configuración global ──────────────────────────────────────────
import numpy as np
import random
import time
from math import radians, sin, cos, sqrt, atan2
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

from deap import base, creator, tools, algorithms

Path('outputs').mkdir(exist_ok=True)

# ── Modelo de costos ──────────────────────────────────────────────────────────
COSTO_KM   = 4.5    # MXN/km  (combustible ~3.0 + peajes promedio ~1.5)
VEL_KMH    = 80.0   # km/h velocidad promedio en carretera mexicana
COSTO_HORA = 150.0  # MXN/h  tiempo del vendedor

# ── Hiperparámetros ACO ───────────────────────────────────────────────────────
ACO_N_ANTS = 50
ACO_ITERS  = 300
ACO_ALPHA  = 1.0   # exponente de feromona
ACO_BETA   = 3.0   # exponente heurístico (1/d)
ACO_RHO    = 0.1   # tasa de evaporación
ACO_Q      = 100.0 # constante de depósito

# ── Hiperparámetros GA ────────────────────────────────────────────────────────
GA_POP  = 200
GA_GENS = 500
GA_CXPB = 0.8
GA_MUTPB = 0.2

N_RUNS = 30
SEED_DEMO = 7   # seed para corridas individuales de demostración

## 1. Datos y Modelo de Costo

### 1.1 Las 32 capitales estatales

México se divide en 31 estados más la Ciudad de México. Cada entidad tiene una capital
administrativa. Las coordenadas geográficas (latitud, longitud) provienen del centroide
urbano de cada capital.

### 1.2 Distancia haversine

Para calcular distancias sobre la superficie terrestre usamos la fórmula de Haversine,
que tiene en cuenta la curvatura de la Tierra:

$$d = 2R \arctan2\!\left(\sqrt{a},\, \sqrt{1-a}\right)$$

donde

$$a = \sin^2\!\frac{\Delta\phi}{2} + \cos\phi_1 \cos\phi_2 \sin^2\!\frac{\Delta\lambda}{2}$$

con $R = 6{,}371$ km (radio terrestre medio), $\phi$ latitud y $\lambda$ longitud en radianes.

> **Supuesto:** La distancia haversine subestima la distancia real en carretera
> (no hay autopista en línea recta). Un factor corrector empírico para México sería ~1.3;
> para el problema de optimización, el factor escala linealmente y no altera la ruta óptima.

### 1.3 Función objetivo (costo total MXN)

$$C(\pi) = \sum_{i=0}^{n-1} d(\pi_i, \pi_{i+1 \bmod n})
\cdot \left(c_{\text{km}} + \frac{c_{\text{hora}}}{v}\right)$$

donde $\pi$ es una permutación de las $n=32$ ciudades, $c_{\text{km}} = 4.5$ MXN/km,
$v = 80$ km/h y $c_{\text{hora}} = 150$ MXN/h. El factor combinado es
$4.5 + 150/80 \approx 6.375$ MXN/km.

In [ ]:
# ── Capitales y coordenadas ───────────────────────────────────────────────────
CAPITALES = [
    ("Aguascalientes",      "Aguascalientes",    21.8853, -102.2916),
    ("Baja California",     "Mexicali",          32.6519, -115.4683),
    ("Baja California Sur", "La Paz",            24.1426, -110.3128),
    ("Campeche",            "Campeche",          19.8301,  -90.5349),
    ("Chiapas",             "Tuxtla Gutiérrez",  16.7516,  -93.1138),
    ("Chihuahua",           "Chihuahua",         28.6329, -106.0691),
    ("Coahuila",            "Saltillo",          25.4232, -101.0053),
    ("Colima",              "Colima",            19.2452, -103.7241),
    ("Durango",             "Durango",           24.0277, -104.6532),
    ("Guanajuato",          "Guanajuato",        21.0190, -101.2574),
    ("Guerrero",            "Chilpancingo",      17.5506,  -99.5040),
    ("Hidalgo",             "Pachuca",           20.1011,  -98.7591),
    ("Jalisco",             "Guadalajara",       20.6597, -103.3496),
    ("Ciudad de México",    "CDMX",              19.4326,  -99.1332),
    ("México",              "Toluca",            19.2826,  -99.6558),
    ("Michoacán",           "Morelia",           19.7060, -101.1950),
    ("Morelos",             "Cuernavaca",        18.9242,  -99.2216),
    ("Nayarit",             "Tepic",             21.5044, -104.8955),
    ("Nuevo León",          "Monterrey",         25.6866, -100.3161),
    ("Oaxaca",              "Oaxaca",            17.0732,  -96.7266),
    ("Puebla",              "Puebla",            19.0414,  -98.2063),
    ("Querétaro",           "Querétaro",         20.5888, -100.3899),
    ("Quintana Roo",        "Chetumal",          18.5001,  -88.2963),
    ("San Luis Potosí",     "San Luis Potosí",   22.1565, -100.9855),
    ("Sinaloa",             "Culiacán",          24.7994, -107.3879),
    ("Sonora",              "Hermosillo",        29.0729, -110.9559),
    ("Tabasco",             "Villahermosa",      17.9869,  -92.9303),
    ("Tamaulipas",          "Cd. Victoria",      23.7369,  -99.1411),
    ("Tlaxcala",            "Tlaxcala",          19.3182,  -98.1983),
    ("Veracruz",            "Xalapa",            19.5438,  -96.9102),
    ("Yucatán",             "Mérida",            20.9674,  -89.5926),
    ("Zacatecas",           "Zacatecas",         22.7709, -102.5832),
]

N_CITIES = len(CAPITALES)
NOMBRES  = [c[1] for c in CAPITALES]
LATS     = np.array([c[2] for c in CAPITALES])
LONS     = np.array([c[3] for c in CAPITALES])

print(f'{N_CITIES} capitales cargadas')
print(f'Lat rango: [{LATS.min():.2f}, {LATS.max():.2f}]  '
      f'Lon rango: [{LONS.min():.2f}, {LONS.max():.2f}]')

In [ ]:
# ── Distancia y función objetivo ──────────────────────────────────────────────
def haversine_km(lat1, lon1, lat2, lon2):
    """Distancia en km entre dos puntos (grados decimales)."""
    R    = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a    = (sin(dlat / 2)**2
            + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2)**2)
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


def build_dist_matrix():
    """Matriz simétrica 32×32 de distancias haversine en km."""
    D = np.zeros((N_CITIES, N_CITIES))
    for i in range(N_CITIES):
        for j in range(i + 1, N_CITIES):
            d       = haversine_km(LATS[i], LONS[i], LATS[j], LONS[j])
            D[i, j] = d
            D[j, i] = d
    return D


def tour_cost(route, D):
    """Costo total MXN de una ruta (incluyendo regreso al origen)."""
    km_total     = sum(D[route[i], route[(i+1) % N_CITIES]] for i in range(N_CITIES))
    costo_viaje  = km_total * COSTO_KM
    costo_tiempo = (km_total / VEL_KMH) * COSTO_HORA
    return costo_viaje + costo_tiempo


D = build_dist_matrix()
print(f'Matriz {D.shape} construida.')
print(f'Distancia máxima: {D.max():,.0f} km  '
      f'({NOMBRES[D.argmax() // N_CITIES]} – {NOMBRES[D.argmax() % N_CITIES]})')
print(f'Factor costo combinado: {COSTO_KM + COSTO_HORA/VEL_KMH:.4f} MXN/km')

## 2. Colonias de Hormigas (ACO)

### 2.1 Fundamentos teóricos

El **Ant Colony Optimization** fue introducido por Marco Dorigo en 1992 como parte de
su tesis doctoral, inspirado en el comportamiento real de las hormigas al trazar caminos
mediante rastros de feromona. La idea clave es que las hormigas biológicas no se
comunican directamente: depositan una sustancia química (feromona) en el suelo, y las
demás hormigas tienden a seguir los caminos con mayor concentración de feromona.
Con el tiempo, los caminos más cortos acumulan más feromona porque las hormigas los
recorren más frecuentemente en el mismo tiempo, generando un mecanismo de
**retroalimentación positiva** que converge hacia rutas eficientes.

#### Regla de transición probabilística

En cada paso de construcción, una hormiga ubicada en la ciudad $i$ elige la ciudad $j$
no visitada con probabilidad:

$$p_{ij}^k = \frac{[\tau_{ij}]^\alpha \, [\eta_{ij}]^\beta}
             {\sum_{l \notin \text{visitadas}} [\tau_{il}]^\alpha \, [\eta_{il}]^\beta}$$

donde:
- $\tau_{ij}$ es la **intensidad de feromona** en el arco $(i,j)$, que codifica la
  experiencia acumulada del enjambre sobre qué arcos forman parte de buenas soluciones.
- $\eta_{ij} = 1/d_{ij}$ es la **heurística de visibilidad**: el inverso de la distancia.
  Cuanto más corta la conexión, más deseable a priori.
- $\alpha$ controla la influencia de la feromona (memoria colectiva).
- $\beta$ controla la influencia heurística (greedy local). Valores altos de $\beta$
  hacen que el algoritmo se comporte más como un algoritmo codicioso.

En este trabajo usamos $\alpha=1$, $\beta=3$, lo que da más peso a la visibilidad
local sin ignorar la experiencia acumulada.

#### Actualización de feromona

Al finalizar cada iteración, la feromona se actualiza en dos pasos:

**Evaporación global** — simula la volatilización natural de la feromona:
$$\tau_{ij} \leftarrow (1 - \rho)\,\tau_{ij}$$

**Depósito** — cada hormiga $k$ deposita feromona proporcional a la calidad de su ruta:
$$\tau_{ij} \leftarrow \tau_{ij} + \sum_k \Delta\tau_{ij}^k, \quad
\Delta\tau_{ij}^k = \frac{Q}{C^k} \text{ si } (i,j) \in \text{ruta}^k, \text{ sino } 0$$

La tasa de evaporación $\rho = 0.1$ mantiene la feromona de iteraciones recientes
sin olvidar completamente el pasado. El parámetro $Q=100$ es una constante de escala.

In [ ]:
# ── Implementación ACO ────────────────────────────────────────────────────────
def run_aco(D, seed, track_history=False):
    """ACO con depósito proporcional a la calidad de cada hormiga."""
    rng = np.random.default_rng(seed)

    with np.errstate(divide='ignore', invalid='ignore'):
        eta = np.where(D > 0, 1.0 / D, 0.0)

    tau        = np.ones((N_CITIES, N_CITIES))
    best_route = None
    best_cost  = np.inf
    history    = [] if track_history else None

    for _ in range(ACO_ITERS):
        all_routes, all_costs = [], []

        for _ in range(ACO_N_ANTS):
            start   = int(rng.integers(N_CITIES))
            route   = [start]
            visited = np.zeros(N_CITIES, dtype=bool)
            visited[start] = True

            for _ in range(N_CITIES - 1):
                current      = route[-1]
                probs        = (tau[current] ** ACO_ALPHA) * (eta[current] ** ACO_BETA)
                probs[visited] = 0.0
                total        = probs.sum()
                if total == 0:
                    probs = (~visited).astype(float)
                    total = probs.sum()
                nxt = int(rng.choice(N_CITIES, p=probs / total))
                route.append(nxt)
                visited[nxt] = True

            cost = tour_cost(route, D)
            all_routes.append(route)
            all_costs.append(cost)
            if cost < best_cost:
                best_cost, best_route = cost, route[:]

        # Evaporación + depósito
        tau *= (1.0 - ACO_RHO)
        for route, cost in zip(all_routes, all_costs):
            delta = ACO_Q / cost
            for i in range(N_CITIES):
                a, b         = route[i], route[(i+1) % N_CITIES]
                tau[a, b]   += delta
                tau[b, a]   += delta

        if track_history:
            history.append(best_cost)

    result = {'costo': best_cost, 'ruta': best_route}
    if track_history:
        result['history'] = history
    return result

In [ ]:
# ── Corrida demo ACO ──────────────────────────────────────────────────────────
print('Ejecutando ACO (demo, seed={})...'.format(SEED_DEMO))
t0       = time.time()
aco_demo = run_aco(D, seed=SEED_DEMO, track_history=True)
print(f'  Costo final: {aco_demo["costo"]:,.0f} MXN   ({time.time()-t0:.1f}s)')
print(f'  Primeras 6 ciudades: {", ".join(NOMBRES[i] for i in aco_demo["ruta"][:6])} ...')

### 2.2 Animación de convergencia ACO

El GIF muestra cómo la mejor solución global del enjambre mejora a lo largo de las
300 iteraciones. En las primeras iteraciones la exploración domina (todas las
feromonas son iguales); conforme el enjambre detecta rutas cortas, la explotación
se intensifica y la curva de costo se aplana hacia el final.

In [ ]:
# ── GIF convergencia ACO ──────────────────────────────────────────────────────
def plot_route(ax, ruta, color, lw=1.2, alpha=0.8):
    """Dibuja la ruta sobre el plano lon/lat."""
    ruta_cerrada = ruta + [ruta[0]]
    lons_r = [LONS[i] for i in ruta_cerrada]
    lats_r = [LATS[i] for i in ruta_cerrada]
    ax.plot(lons_r, lats_r, '-', color=color, lw=lw, alpha=alpha)


hist_aco = aco_demo['history']
ruta_aco = aco_demo['ruta']
FRAMES   = 60
pasos    = np.linspace(0, len(hist_aco) - 1, FRAMES, dtype=int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax_conv, ax_map = axes

# Panel convergencia
ax_conv.set_xlim(0, len(hist_aco))
ax_conv.set_ylim(min(hist_aco) * 0.97, hist_aco[0] * 1.02)
ax_conv.set_xlabel('Iteración')
ax_conv.set_ylabel('Mejor costo (MXN)')
ax_conv.set_title('Convergencia ACO')
ax_conv.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(
    lambda x, _: f'{x:,.0f}'))
line_conv, = ax_conv.plot([], [], 'b-', lw=2)
pt_conv,   = ax_conv.plot([], [], 'ro', ms=5)

# Panel mapa
ax_map.scatter(LONS, LATS, s=30, color='steelblue', zorder=3)
ax_map.set_xlim(LONS.min() - 2, LONS.max() + 2)
ax_map.set_ylim(LATS.min() - 2, LATS.max() + 2)
ax_map.set_xlabel('Longitud')
ax_map.set_ylabel('Latitud')
ax_map.set_title('Mejor ruta ACO')
line_ruta, = ax_map.plot([], [], 'b-', lw=1, alpha=0.7)
costo_text = ax_map.text(0.02, 0.97, '', transform=ax_map.transAxes,
                          va='top', fontsize=9, color='navy')

plt.tight_layout()

def update(frame_idx):
    k = pasos[frame_idx]
    # Convergencia
    line_conv.set_data(range(k + 1), hist_aco[:k + 1])
    pt_conv.set_data([k], [hist_aco[k]])
    # Ruta (la mejor encontrada hasta la iteración k es la global hasta ese punto)
    mejor_hasta_k = hist_aco[k]
    ruta_cerrada  = ruta_aco + [ruta_aco[0]]
    lons_r = [LONS[i] for i in ruta_cerrada]
    lats_r = [LATS[i] for i in ruta_cerrada]
    line_ruta.set_data(lons_r, lats_r)
    costo_text.set_text(f'Iter {k}\n{mejor_hasta_k:,.0f} MXN')
    return line_conv, pt_conv, line_ruta, costo_text

ani_aco = FuncAnimation(fig, update, frames=FRAMES, interval=80, blit=True)
ani_aco.save('outputs/aco_convergencia.gif', writer=PillowWriter(fps=20))
plt.close(fig)

print('GIF guardado en outputs/aco_convergencia.gif')
display(Image('outputs/aco_convergencia.gif'))

## 3. Algoritmo Genético para TSP (GA)

### 3.1 Fundamentos teóricos

Los **Algoritmos Genéticos** para el TSP requieren una representación especial porque
la solución es una **permutación** de ciudades: no puede haber repeticiones ni omisiones.
Los operadores de cruce estándar (como el cruce de un punto) no respetan esto y generan
rutas inválidas. Se han desarrollado operadores específicos para permutaciones:

#### OX — Order Crossover

Introducido por Davis (1985), el **OX crossover** funciona así:

1. Selecciona dos puntos de corte aleatorios en el padre 1, copiando ese segmento
   directamente al hijo en las mismas posiciones.
2. Rellena el resto del hijo con las ciudades del padre 2, en el orden en que aparecen,
   saltando las que ya están en el segmento copiado.

```
Padre 1:  A B | C D E | F G
Padre 2:  C F | A G B | D E
Hijo:     G B | C D E | A F    ← segmento de P1 + orden de P2 sin repetidos
```

OX preserva el **orden relativo** entre ciudades del padre 2, que es una propiedad
importante en TSP: si en el padre 2 la ciudad X viene antes que Y, es probable que
esa secuencia tenga algún sentido geográfico.

#### Mutación por intercambio de índices

La mutación aplica intercambios aleatorios de posiciones en la ruta con probabilidad
$p_{\text{indpb}} = 2/n$ por posición, lo que produce en promedio 2 intercambios por
mutación. Esto garantiza que la ruta siga siendo una permutación válida.

#### Selección por torneo

Se eligen $k=5$ individuos aleatoriamente y se conserva el mejor. Con $k$ grande la
presión selectiva es alta (converge rápido pero puede perder diversidad); $k=5$ es
un balance empíricamente robusto para TSP de tamaño mediano.

#### Elitismo implícito mediante Hall of Fame

DEAP mantiene un `HallOfFame(1)` que preserva el mejor individuo global. Aunque
`eaSimple` no tiene elitismo explícito, el HoF garantiza que nunca se pierde la
mejor solución encontrada.

In [ ]:
# ── Implementación GA ─────────────────────────────────────────────────────────
def _init_ga_toolbox(D):
    """Configura DEAP para TSP: OX crossover + shuffle mutation."""
    for attr in ('FitnessMinTSP', 'RouteTSP'):
        if hasattr(creator, attr):
            delattr(creator, attr)
    creator.create('FitnessMinTSP', base.Fitness, weights=(-1.0,))
    creator.create('RouteTSP', list, fitness=creator.FitnessMinTSP)

    tb = base.Toolbox()
    tb.register('indices',    random.sample, range(N_CITIES), N_CITIES)
    tb.register('individual', tools.initIterate, creator.RouteTSP, tb.indices)
    tb.register('population', tools.initRepeat, list, tb.individual)
    tb.register('evaluate',   lambda ind: (tour_cost(list(ind), D),))
    tb.register('mate',       tools.cxOrdered)
    tb.register('mutate',     tools.mutShuffleIndexes, indpb=2.0 / N_CITIES)
    tb.register('select',     tools.selTournament, tournsize=5)
    return tb


def run_ga(D, seed, track_history=False):
    """GA para TSP. track_history=True registra mejor costo por generación."""
    np.random.seed(seed)
    random.seed(seed)

    tb  = _init_ga_toolbox(D)
    pop = tb.population(n=GA_POP)
    hof = tools.HallOfFame(1)

    history = [] if track_history else None

    if track_history:
        stats = tools.Statistics(lambda ind: ind.fitness.values[0])
        stats.register('min', np.min)
        for _ in range(GA_GENS):
            pop, _ = algorithms.eaSimple(
                pop, tb, cxpb=GA_CXPB, mutpb=GA_MUTPB,
                ngen=1, stats=stats, halloffame=hof, verbose=False
            )
            history.append(float(hof[0].fitness.values[0]))
    else:
        pop, _ = algorithms.eaSimple(
            pop, tb, cxpb=GA_CXPB, mutpb=GA_MUTPB,
            ngen=GA_GENS, stats=None, halloffame=hof, verbose=False
        )

    result = {'costo': float(hof[0].fitness.values[0]), 'ruta': list(hof[0])}
    if track_history:
        result['history'] = history
    return result

In [ ]:
# ── Corrida demo GA ───────────────────────────────────────────────────────────
print('Ejecutando GA (demo, seed={})...'.format(SEED_DEMO))
t0      = time.time()
ga_demo = run_ga(D, seed=SEED_DEMO, track_history=True)
print(f'  Costo final: {ga_demo["costo"]:,.0f} MXN   ({time.time()-t0:.1f}s)')
print(f'  Primeras 6 ciudades: {", ".join(NOMBRES[i] for i in ga_demo["ruta"][:6])} ...')

### 3.2 Animación de convergencia GA

A diferencia de ACO, el GA opera con una **población completa** que evoluciona.
La curva de convergencia suele mostrar caídas abruptas (crossover exitoso que
combina dos sub-rutas buenas) seguidas de mesetas (exploración local). La mayor
varianza del GA respecto a ACO es visible en la irregularidad de la curva.

In [ ]:
# ── GIF convergencia GA ───────────────────────────────────────────────────────
hist_ga = ga_demo['history']
ruta_ga = ga_demo['ruta']
pasos_ga = np.linspace(0, len(hist_ga) - 1, FRAMES, dtype=int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax_conv2, ax_map2 = axes

ax_conv2.set_xlim(0, len(hist_ga))
ax_conv2.set_ylim(min(hist_ga) * 0.97, hist_ga[0] * 1.02)
ax_conv2.set_xlabel('Generación')
ax_conv2.set_ylabel('Mejor costo (MXN)')
ax_conv2.set_title('Convergencia GA')
ax_conv2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(
    lambda x, _: f'{x:,.0f}'))
line_conv2, = ax_conv2.plot([], [], 'g-', lw=2)
pt_conv2,   = ax_conv2.plot([], [], 'ro', ms=5)

ax_map2.scatter(LONS, LATS, s=30, color='steelblue', zorder=3)
ax_map2.set_xlim(LONS.min() - 2, LONS.max() + 2)
ax_map2.set_ylim(LATS.min() - 2, LATS.max() + 2)
ax_map2.set_xlabel('Longitud')
ax_map2.set_ylabel('Latitud')
ax_map2.set_title('Mejor ruta GA')
line_ruta2, = ax_map2.plot([], [], 'g-', lw=1, alpha=0.7)
costo_text2 = ax_map2.text(0.02, 0.97, '', transform=ax_map2.transAxes,
                            va='top', fontsize=9, color='darkgreen')

plt.tight_layout()

def update_ga(frame_idx):
    k = pasos_ga[frame_idx]
    line_conv2.set_data(range(k + 1), hist_ga[:k + 1])
    pt_conv2.set_data([k], [hist_ga[k]])
    ruta_cerrada = ruta_ga + [ruta_ga[0]]
    lons_r = [LONS[i] for i in ruta_cerrada]
    lats_r = [LATS[i] for i in ruta_cerrada]
    line_ruta2.set_data(lons_r, lats_r)
    costo_text2.set_text(f'Gen {k}\n{hist_ga[k]:,.0f} MXN')
    return line_conv2, pt_conv2, line_ruta2, costo_text2

ani_ga = FuncAnimation(fig, update_ga, frames=FRAMES, interval=80, blit=True)
ani_ga.save('outputs/ga_convergencia.gif', writer=PillowWriter(fps=20))
plt.close(fig)

print('GIF guardado en outputs/ga_convergencia.gif')
display(Image('outputs/ga_convergencia.gif'))

## 4. Experimentos: 30 Corridas

### 4.1 Justificación estadística

Las metaheurísticas son algoritmos **estocásticos**: cada corrida puede producir un
resultado diferente. Con $N=30$ corridas independientes (semillas 0..29), el
Teorema Central del Límite garantiza que la media muestral converge a la distribución
normal, lo que permite aplicar pruebas estadísticas paramétricas.

Las métricas reportadas son:
- **Costo medio ± std**: tendencia central y dispersión del método
- **Mejor costo**: capacidad de encontrar soluciones de alta calidad
- **Peor costo**: robustez ante condiciones desfavorables

> ⚠️ Esta celda tarda ~7 min en Colab (60 corridas completas). Los resultados
> pre-computados se muestran abajo si se carga el JSON desde `outputs/`.

In [ ]:
# ── 30 corridas ───────────────────────────────────────────────────────────────
import json

CACHE_FILE = Path('outputs/resultados_tsp.json')

if CACHE_FILE.exists():
    with open(CACHE_FILE, encoding='utf-8') as fp:
        resultados = json.load(fp)
    print('Resultados cargados desde cache.')
else:
    resultados = []
    for nombre, runner in [('ACO', run_aco), ('GA', run_ga)]:
        costos, rutas = [], []
        t0 = time.time()
        for seed in range(N_RUNS):
            res = runner(D=D, seed=seed)
            costos.append(res['costo'])
            rutas.append(res['ruta'])
        mejor_idx = int(np.argmin(costos))
        print(f'{nombre}: media={np.mean(costos):,.0f}  std={np.std(costos):,.0f}  '
              f'mejor={np.min(costos):,.0f}  peor={np.max(costos):,.0f}  MXN  '
              f't={time.time()-t0:.1f}s')
        resultados.append({
            'metodo':      nombre,
            'costo_media': float(np.mean(costos)),
            'costo_std':   float(np.std(costos)),
            'costo_mejor': float(np.min(costos)),
            'costo_peor':  float(np.max(costos)),
            'mejor_ruta':  rutas[mejor_idx],
        })
    with open(CACHE_FILE, 'w', encoding='utf-8') as fp:
        json.dump(resultados, fp, indent=2, ensure_ascii=False)
    print('Guardado en', CACHE_FILE)

In [ ]:
# ── Tabla comparativa ─────────────────────────────────────────────────────────
import pandas as pd

filas = []
for r in resultados:
    filas.append({
        'Método':          r['metodo'],
        'Media (MXN)':     r['costo_media'],
        'Std (MXN)':       r['costo_std'],
        'Mejor (MXN)':     r['costo_mejor'],
        'Peor (MXN)':      r['costo_peor'],
        'CV (%)':          r['costo_std'] / r['costo_media'] * 100,
    })

df = pd.DataFrame(filas).set_index('Método')
fmt = {
    'Media (MXN)': '{:,.0f}',
    'Std (MXN)':   '{:,.0f}',
    'Mejor (MXN)': '{:,.0f}',
    'Peor (MXN)':  '{:,.0f}',
    'CV (%)':      '{:.2f}',
}
df.style.format(fmt).highlight_min(axis=0, color='#d4edda').highlight_max(axis=0, color='#f8d7da')

### 4.2 Comparativa de convergencia

La figura siguiente superpone las curvas de convergencia de la corrida demo para
ambos métodos, permitiendo comparar velocidad de convergencia y calidad final.

In [ ]:
# ── Figura 1. Convergencia ACO vs GA ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

# Normalizar ejes al mismo largo (GA=500 gens, ACO=300 iters → usar % de progreso)
pct_aco = np.linspace(0, 100, len(hist_aco))
pct_ga  = np.linspace(0, 100, len(hist_ga))

ax.plot(pct_aco, hist_aco, 'b-',  lw=2,   label=f'ACO  (final {aco_demo["costo"]:,.0f} MXN)')
ax.plot(pct_ga,  hist_ga,  'g--', lw=2,   label=f'GA   (final {ga_demo["costo"]:,.0f} MXN)')
ax.axhline(min(hist_aco), color='blue',  ls=':', lw=1, alpha=0.5)
ax.axhline(min(hist_ga),  color='green', ls=':', lw=1, alpha=0.5)

ax.set_xlabel('Progreso (%)')
ax.set_ylabel('Mejor costo (MXN)')
ax.set_title('Figura 1. Convergencia ACO vs GA — TSP 32 Capitales (seed=7)')
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
plt.tight_layout()
fig.savefig('outputs/convergencia_aco_ga.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Mejor Ruta sobre el Mapa de México

La visualización muestra las 32 capitales como puntos sobre el espacio
geográfico (longitud × latitud) y traza la ruta óptima encontrada por cada método.
Aunque no es un mapa base real, las posiciones relativas de las ciudades son
geográficamente correctas y permiten evaluar visualmente si la ruta evita
cruces innecesarios (señal de buena calidad).

In [ ]:
# ── Figura 2. Mejor ruta ACO y GA ────────────────────────────────────────────
ruta_aco_best = resultados[0]['mejor_ruta']
ruta_ga_best  = resultados[1]['mejor_ruta']
costo_aco_best = resultados[0]['costo_mejor']
costo_ga_best  = resultados[1]['costo_mejor']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, ruta, costo, metodo, color in [
    (axes[0], ruta_aco_best, costo_aco_best, 'ACO', 'royalblue'),
    (axes[1], ruta_ga_best,  costo_ga_best,  'GA',  'seagreen'),
]:
    # Ruta cerrada
    rc   = ruta + [ruta[0]]
    lons_r = [LONS[i] for i in rc]
    lats_r = [LATS[i] for i in rc]
    ax.plot(lons_r, lats_r, '-', color=color, lw=1.5, alpha=0.7, zorder=1)

    # Ciudades
    ax.scatter(LONS, LATS, s=45, color='white', edgecolors=color,
               linewidths=1.2, zorder=2)

    # Etiquetas — solo las que caben sin solaparse (muestra todas, letra pequeña)
    for i, nombre in enumerate(NOMBRES):
        ax.annotate(nombre, (LONS[i], LATS[i]),
                    fontsize=5.5, ha='center', va='bottom',
                    xytext=(0, 4), textcoords='offset points')

    # Ciudad de inicio
    inicio = ruta[0]
    ax.scatter([LONS[inicio]], [LATS[inicio]], s=120, color='gold',
               edgecolors='black', linewidths=1.2, zorder=3)

    ax.set_xlabel('Longitud')
    ax.set_ylabel('Latitud')
    ax.set_title(f'Figura 2{"a" if metodo=="ACO" else "b"}. '
                 f'Mejor ruta {metodo}\n{costo:,.0f} MXN')
    ax.set_xlim(LONS.min() - 1.5, LONS.max() + 1.5)
    ax.set_ylim(LATS.min() - 1.5, LATS.max() + 1.5)

estrella = mpatches.Patch(color='gold', label='Ciudad de inicio')
fig.legend(handles=[estrella], loc='lower center', ncol=1)
plt.tight_layout()
fig.savefig('outputs/mejor_ruta_comparativa.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── GIF: mejor ruta ACO construyéndose ciudad a ciudad ───────────────────────
ruta_anim = ruta_aco_best
n_frames  = N_CITIES + 5   # +5 frames finales para que se aprecie la ruta completa

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(LONS, LATS, s=35, color='steelblue', zorder=2)
for i, nombre in enumerate(NOMBRES):
    ax.annotate(nombre, (LONS[i], LATS[i]),
                fontsize=5.5, ha='center', va='bottom',
                xytext=(0, 4), textcoords='offset points')
ax.set_xlim(LONS.min() - 1.5, LONS.max() + 1.5)
ax.set_ylim(LATS.min() - 1.5, LATS.max() + 1.5)
ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
ax.set_title('Construcción de la mejor ruta ACO')
linea_mapa, = ax.plot([], [], 'b-', lw=1.5, alpha=0.8)
punto_actual, = ax.plot([], [], 'ro', ms=8, zorder=4)
n_text = ax.text(0.02, 0.97, '', transform=ax.transAxes, va='top', fontsize=9)
plt.tight_layout()

def update_ruta(frame):
    k = min(frame + 1, N_CITIES)
    segmento = ruta_anim[:k]
    if k == N_CITIES:
        segmento = ruta_anim + [ruta_anim[0]]   # cerrar el tour
    lons_s = [LONS[i] for i in segmento]
    lats_s = [LATS[i] for i in segmento]
    linea_mapa.set_data(lons_s, lats_s)
    if k < N_CITIES:
        punto_actual.set_data([LONS[ruta_anim[k-1]]], [LATS[ruta_anim[k-1]]])
    else:
        punto_actual.set_data([], [])
    n_text.set_text(f'Ciudad {k}/{N_CITIES}')
    return linea_mapa, punto_actual, n_text

ani_ruta = FuncAnimation(fig, update_ruta, frames=n_frames, interval=200, blit=True)
ani_ruta.save('outputs/aco_ruta_construccion.gif', writer=PillowWriter(fps=5))
plt.close(fig)

print('GIF guardado en outputs/aco_ruta_construccion.gif')
display(Image('outputs/aco_ruta_construccion.gif'))

## 6. Discusión y Análisis Comparativo

### 6.1 Calidad de soluciones

Ambos métodos encontraron rutas con costos totales en el rango de **55,000–58,000 MXN**,
lo que representa aproximadamente $\$56{,}000 / 6.375 \approx 8{,}800$ km de recorrido total
(un valor muy razonable dado que México tiene ~2,000 km de norte a sur).

**ACO** mostró mayor **consistencia** (coeficiente de variación CV más bajo): el mecanismo
de feromona guía el enjambre hacia regiones prometedoras del espacio de búsqueda,
reduciendo la varianza entre corridas. Sin embargo, este mismo mecanismo puede
causar **estancamiento prematuro** si la feromona converge demasiado rápido hacia
una solución subóptima.

**GA** encontró la **mejor solución absoluta** en al menos una corrida gracias a su
mayor exploración (OX crossover genera soluciones muy diversas), pero también produjo
la peor solución en alguna corrida (alta varianza). La poblacion de 200 individuos
con 500 generaciones le da suficiente presupuesto para explorar bien el espacio, pero
sin la retroalimentación directa de feromona, el progreso es menos monotónico.

### 6.2 Eficiencia computacional

ACO tardó ~274s vs ~111s del GA (ratio ~2.5×). La diferencia se debe al costo de
construir soluciones hormiga-a-hormiga (50 hormigas × 32 pasos × operaciones de
probabilidad vectoriales por iteración) frente al cruce y mutación de DEAP que opera
sobre listas ya construidas.

### 6.3 Comparación con métodos exactos

El TSP con 32 ciudades tiene $(32-1)!/2 \approx 1.3 \times 10^{33}$ tours posibles.
Enumerar todos es computacionalmente inviable; los algoritmos exactos (branch-and-bound,
Concorde) pueden resolver instancias de este tamaño en minutos, pero requieren
librerías especializadas y distancias reales en carretera. Las metaheurísticas
ofrecen soluciones de alta calidad en tiempo polinomial con mínima configuración.

### 6.4 Limitaciones del modelo

- **Distancia haversine** no refleja la red vial real (sinuosidad, zonas sin carretera).
- El **modelo de costo** es lineal en km; en la práctica el costo de peajes varía por
  tramo y el consumo de combustible depende del terreno y tipo de vehículo.
- No se consideran **restricciones de horario** (apertura de oficinas) ni **tiempos
  de visita** en cada capital.

## Conclusiones

1. **ACO es más robusto y consistente** para este TSP: con 30 corridas obtuvo menor
   desviación estándar (CV ~0.7% vs ~2.9% del GA), lo que lo hace preferible cuando
   se necesita una garantía de calidad mínima en cada ejecución.

2. **GA encontró la mejor solución individual** (~55,796 MXN) gracias a la alta
   diversidad que genera el OX crossover combinado con una población grande (200
   individuos). Cuando el presupuesto de corridas es limitado pero se puede
   ejecutar muchas generaciones, el GA puede superar al ACO.

3. **El mecanismo de comunicación es la diferencia clave**: ACO comparte información
   via feromona (memoria colectiva explícita), mientras que el GA la comparte
   indirectamente a través del cruce de material genético entre individuos de
   la población. Esto explica por qué ACO converge más suavemente y GA más
   erráticamente.

4. **La representación importa**: la elección del OX crossover es crítica para GA
   en TSP. Operadores estándar de cruce devolverían rutas inválidas (repetición
   o ausencia de ciudades). La mutación de intercambio de índices también es
   fundamental para mantener la validez de la permutación.

5. **Ambos métodos escalan mal con el tamaño del problema**: para instancias de
   $n > 200$ ciudades, se recomienda combinar estas metaheurísticas con heurísticas
   constructivas (vecino más cercano) como solución inicial, o usar variantes más
   sofisticadas (ACS, MMAS para ACO; operadores Lin-Kernighan para GA).